In [1]:
import os
import numpy as np
from scipy.io import wavfile
from pydub import AudioSegment

c:\Users\reach\Projects\copd-audio-ml\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [ ]:
segments_dir = "data/processed/segments"
preprocess_dir = "data/preprocessed"
classes = ["copd", "healthy"]

In [ ]:
target_length = 16000 * 2  #2 seconds at 16kHz

#Create preprocessed folders
for cls in classes:
    os.makedirs(os.path.join(preprocess_dir, cls), exist_ok=True)

In [ ]:
def preprocess_segment(wav_path, target_length=target_length):
    # oad audio
    sr, audio = wavfile.read(wav_path)
    
    #Convert to mono
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    
    #Normalize amplitude
    audio = audio / np.max(np.abs(audio))
    
    #Pad or clip to target length
    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)), mode="constant")
    else:
        audio = audio[:target_length]
    
    return audio, sr

In [ ]:
#Process all segments
for cls in classes:
    cls_in_dir = os.path.join(segments_dir, cls)
    cls_out_dir = os.path.join(preprocess_dir, cls)
    
    for fname in os.listdir(cls_in_dir):
        if not fname.endswith(".wav"):
            continue
        
        in_path = os.path.join(cls_in_dir, fname)
        audio_proc, sr = preprocess_segment(in_path)
        
        # Save as .npy for ML
        out_path = os.path.join(cls_out_dir, fname.replace(".wav", ".npy"))
        np.save(out_path, audio_proc)
        
#Preprocessed segments stored as .npy in data/preprocessed


Preprocessing complete. Preprocessed segments stored as .npy in data/preprocessed/
